# Experimentación de modelos
La implementación completa vive en clases OOP y este notebook documenta y ejecuta la comparación.


In [ ]:
from volume_forecasting.data_loader import DataLoader
from volume_forecasting.feature_engineering import FeatureEngineer
from volume_forecasting.experiment import ForecastExperiment
from volume_forecasting.artifacts import ArtifactStore


In [ ]:
volumen, variables_macro = DataLoader().load_sources('../data/raw/Volume Data_prueba_CD.xlsx', '../data/raw/Macro Variables_Prueba_CD.xlsx')
data = FeatureEngineer().build(volumen, variables_macro)


In [ ]:
feature_sets = {
    'compact': ['Volume_lag_1','DN_lag_1','Avg_Price_lag_1','Is_December','Low_Season'],
    'initial': ['Volume_lag_1','DN_lag_1','Avg_Price_lag_1','IPC_lag_1','Is_December','Low_Season'],
    'moving_average': ['Volume_MA_3','DN_lag_1','Avg_Price_lag_1','IPC_lag_1','Is_December','Low_Season'],
}
experiment = ForecastExperiment(feature_sets, test_months=12, max_abs_bias=0.07)
output = experiment.run(data)
display(output.results)
display(output.best_result)


## Exportar el ganador
Se guarda el pipeline completo y el contrato JSON.


In [ ]:
best = output.best_result
metadata = {
    'model_key': f"{best['Feature_Set']}__{best['Model']}",
    'feature_set': best['Feature_Set'],
    'features': feature_sets[best['Feature_Set']],
    'target': 'Log_Volume',
    'best_params': best['Best_Params'],
    'holdout_metrics': {'wape': best['WAPE'], 'bias': best['BIAS'], 'mae': best['MAE'], 'rmse': best['RMSE']}
}
ArtifactStore.save(output.best_model, metadata, '../models/best_model.joblib', '../models/model_metadata.json')
